In [1]:
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm

In [2]:
df = pd.read_csv('../data/processed/filtered_complaints.csv')
print(df.shape)
df['product_category'].value_counts()

(455037, 9)


product_category
Credit Card        189334
Savings Account    140319
Money Transfer      98685
Personal Loan       26699
Name: count, dtype: int64

In [3]:
SAMPLE_SIZE = 12000

def stratified_sample(df, total_n, strat_col):
    fractions = df[strat_col].value_counts(normalize=True)
    samples = []
    for category, frac in fractions.items():
        n = int(round(frac * total_n))
        cat_df = df[df[strat_col] == category]
        samples.append(cat_df.sample(n=min(n, len(cat_df)), random_state=42))
    return pd.concat(samples).reset_index(drop=True)

sample_df = stratified_sample(df, SAMPLE_SIZE, 'product_category')
print(sample_df.shape)
print(sample_df['product_category'].value_counts())

(11999, 9)
product_category
Credit Card        4993
Savings Account    3700
Money Transfer     2602
Personal Loan       704
Name: count, dtype: int64


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadatas = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text_chunks = splitter.split_text(str(row['cleaned_narrative']))
    for i, chunk in enumerate(text_chunks):
        chunks.append(chunk)
        metadatas.append({
            "complaint_id": str(row['Complaint ID']),
            "product_category": row['product_category'],
            "issue": str(row.get('Issue', '')),
            "company": str(row.get('Company', '')),
            "chunk_index": i,
            "total_chunks": len(text_chunks)
        })

print(f"Total chunks created: {len(chunks)}")

100%|██████████| 11999/11999 [00:05<00:00, 2179.71it/s]

Total chunks created: 35404


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()  

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

c:\Users\HP EliteBook\Desktop\KAIM\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP EliteBook\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384


C:\Users\HP EliteBook\AppData\Local\Temp\ipykernel_12048\566711332.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


In [6]:
embeddings = model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(embeddings.shape)

Batches:   0%|          | 0/554 [00:00<?, ?it/s]

(35404, 384)


In [7]:
client = chromadb.PersistentClient(path="../vector_store")

collection = client.get_or_create_collection(name="complaint_chunks")

ids = [f"chunk_{i}" for i in range(len(chunks))]

batch_size = 500
for i in tqdm(range(0, len(chunks), batch_size)):
    collection.add(
        ids=ids[i:i+batch_size],
        documents=chunks[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size]
    )

print(f"Collection count: {collection.count()}")

100%|██████████| 71/71 [01:39<00:00,  1.40s/it]

Collection count: 35404


In [8]:
test_query = "unauthorized credit card charges"
test_embedding = model.encode([test_query]).tolist()

results = collection.query(
    query_embeddings=test_embedding,
    n_results=5
)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(meta['product_category'], '|', meta['complaint_id'])
    print(doc[:150], '\n')

Credit Card | 3374042
. i handed the card over to merchants on several occasions for them to run it through for other charges that i recognize like for a dinner or gas . so 

Credit Card | 3880169
on 2020, i received an unauthorized charge on my credit card. i immediately contacted bank and cancelled my card and ordered a new one. 5 3 credited t 

Credit Card | 8012052
an unauthorized charge from appeared on my credit card statement in of 2023. i have made attempts with the flight school and spent many hours in talks 

Credit Card | 7912467
. if you recognize all the other charges, you don't need to do anything else. if you see other charges that aren't yours, please call us using the num 

Savings Account | 9222991
. the following charges were not authorized by me and were made when my card was stolen 50.00 28.00 t 100.00 320.00 200.00 28.00 210.00 550.00 580.00 

